# Olist Brazilian E-Commerce
## Complete Python EDA + Business Analysis

This notebook contains the complete Olist E-Commerce exploratory data analysis and business analysis workflow.

**Sections covered**
1. Project / Data Location
2. File Discovery
3. Data Loading
4. Dataset Overview
5. Data Quality Checks
6. Date Analysis
7. Key / Cardinality Analysis
8. Geography & Referential Integrity
9. Master Dataset
10. Sales & Business Analysis
11. Customer Analysis
12. Seller Analysis
13. Payment Analysis
14. Review Analysis
15. Delivery & Logistics Analysis
16. Freight & Outlier Analysis
17. Geographic Analysis
18. Correlation Analysis
19. Final KPI Summary
20. Visualizations


## 0. Import Libraries

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 1. Project / Data Location

Jupyter Notebook does not reliably provide `__file__`, so the notebook uses the current working directory. Update `PROJECT_DIR` if your Olist folder is elsewhere.

In [ ]:
# Option 1: If the notebook is inside the Olist_Ecommerce project folder
PROJECT_DIR = Path.cwd()

# Option 2: Manually specify your project folder:
# PROJECT_DIR = Path(r"C:\Users\YourName\Desktop\Olist_Ecommerce")

print("=" * 80)
print("OLIST BRAZILIAN E-COMMERCE — PYTHON EDA")
print("=" * 80)
print("\nCurrent working directory:")
print(Path.cwd())
print("\nProject folder:")
print(PROJECT_DIR.resolve())


## 2. Find Files Automatically

In [ ]:
def find_file(filename):
    matches = list(PROJECT_DIR.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"\nCould not find '{filename}' inside:\n"
            f"{PROJECT_DIR}\n\n"
            f"Please make sure the file exists inside the Olist_Ecommerce folder."
        )

    if len(matches) > 1:
        print(f"\nWarning: Multiple files found for {filename}")
        for match in matches:
            print(" -", match)
        print("Using:", matches[0])

    return matches[0]


## 3. Locate Data Files

In [ ]:
orders_path = find_file("olist_orders_dataset.csv")
order_items_path = find_file("olist_order_items_dataset.csv")
customers_path = find_file("olist_customers_dataset.csv")
products_path = find_file("olist_products_dataset.csv")
payments_path = find_file("olist_order_payments_dataset.csv")
reviews_path = find_file("olist_order_reviews_dataset.csv")
sellers_path = find_file("olist_sellers_dataset.csv")
geolocation_path = find_file("olist_geolocation_dataset.csv")
category_translation_path = find_file("product_category_name_translation.csv")

print("All required files located successfully.")


## 4. Read Data

In [ ]:
print("\n" + "=" * 80)
print("READING DATA")
print("=" * 80)

orders = pd.read_csv(orders_path)
order_items = pd.read_csv(order_items_path)
customers = pd.read_csv(customers_path)
products = pd.read_csv(products_path)
payments = pd.read_csv(payments_path)
reviews = pd.read_csv(reviews_path)
sellers = pd.read_csv(sellers_path)
geolocation = pd.read_csv(geolocation_path)
category_translation = pd.read_csv(category_translation_path)

print("Data loaded successfully.")


## 5. Dataset Dictionary

In [ ]:
datasets = {
    "Orders": orders,
    "Order Items": order_items,
    "Customers": customers,
    "Products": products,
    "Payments": payments,
    "Reviews": reviews,
    "Sellers": sellers,
    "Geolocation": geolocation,
    "Category Translation": category_translation
}


## 6. Data Loading Summary

In [ ]:
print("\n" + "=" * 80)
print("DATA LOADED SUCCESSFULLY")
print("=" * 80)

for name, df in datasets.items():
    print(
        f"{name:<25}"
        f"Rows: {len(df):>10,} | "
        f"Columns: {df.shape[1]:>3}"
    )


## 7. Dataset Overview

In [ ]:
for name, df in datasets.items():
    print("\n" + "-" * 70)
    print(name)
    print("-" * 70)
    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())


## 8. Data Types

In [ ]:
for name, df in datasets.items():
    print("\n" + "-" * 70)
    print(name)
    print(df.dtypes)


## 9. Missing Values

In [ ]:
for name, df in datasets.items():
    missing_count = df.isnull().sum()
    missing_percentage = (missing_count / len(df) * 100).round(2)

    missing_table = pd.DataFrame({
        "missing_count": missing_count,
        "missing_percentage": missing_percentage
    })

    missing_table = missing_table[missing_table["missing_count"] > 0]

    print("\n" + "-" * 70)
    print(name)

    if missing_table.empty:
        print("No missing values.")
    else:
        display(missing_table.sort_values("missing_count", ascending=False))


## 10. Complete Duplicate Rows

In [ ]:
print("\n" + "=" * 80)
print("COMPLETE DUPLICATE ROWS")
print("=" * 80)

for name, df in datasets.items():
    print(f"{name:<25}: {df.duplicated().sum():,}")


## 11. Date Conversion

In [ ]:
date_columns = {
    "Orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "Order Items": ["shipping_limit_date"],
    "Reviews": ["review_creation_date", "review_answer_timestamp"]
}

for name, columns in date_columns.items():
    for column in columns:
        datasets[name][column] = pd.to_datetime(
            datasets[name][column],
            errors="coerce"
        )

print("Date conversion completed.")


## 12. Date Ranges

In [ ]:
for name, columns in date_columns.items():
    print("\n" + name)
    for column in columns:
        series = datasets[name][column].dropna()
        if len(series) > 0:
            print(f"{column:<40}{series.min()}  →  {series.max()}")


## 13. Primary Key Duplicate Check

In [ ]:
primary_keys = {
    "Orders": "order_id",
    "Customers": "customer_id",
    "Products": "product_id",
    "Sellers": "seller_id",
    "Reviews": "review_id"
}

for name, key in primary_keys.items():
    df = datasets[name]
    print(
        f"{name:<20}"
        f"Key: {key:<25}"
        f"Duplicates: {df[key].duplicated().sum():,}"
    )


## 14. Cardinality Analysis

In [ ]:
for name, df in datasets.items():
    print("\n" + name)
    for column in df.columns:
        print(
            f"{column:<45}"
            f"Unique: {df[column].nunique(dropna=True):,}"
        )


## 15. Geolocation Duplicate Analysis

In [ ]:
geo_exact_duplicates = geolocation.duplicated().sum()
geo_duplicate_rows = geolocation.duplicated(keep=False).sum()
geo_zip_unique = geolocation["geolocation_zip_code_prefix"].nunique()

print("Exact duplicate rows:", f"{geo_exact_duplicates:,}")
print("Rows involved in duplicates:", f"{geo_duplicate_rows:,}")
print("Unique ZIP prefixes:", f"{geo_zip_unique:,}")


## 16. Review ID Duplicate Investigation

In [ ]:
review_duplicate_rows = reviews[
    reviews["review_id"].duplicated(keep=False)
]

print("Rows involved in duplicate review IDs:", len(review_duplicate_rows))
print("Duplicate review IDs:", review_duplicate_rows["review_id"].nunique())


## 17. Timestamp Logical Checks

In [ ]:
approval_before_purchase = (
    orders["order_approved_at"] < orders["order_purchase_timestamp"]
).sum()

carrier_before_approval = (
    orders["order_delivered_carrier_date"] < orders["order_approved_at"]
).sum()

customer_before_carrier = (
    orders["order_delivered_customer_date"] <
    orders["order_delivered_carrier_date"]
).sum()

print("Approval before purchase:", approval_before_purchase)
print("Carrier before approval:", carrier_before_approval)
print("Customer delivery before carrier:", customer_before_carrier)


## 18. Numerical Summary

In [ ]:
for name, df in datasets.items():
    numeric_df = df.select_dtypes(include=np.number)

    if not numeric_df.empty:
        print("\n" + name)
        display(numeric_df.describe().round(2))


## 19. Negative Value Check

In [ ]:
numeric_columns_to_check = {
    "Order Items": ["price", "freight_value"],
    "Products": [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ],
    "Payments": ["payment_value", "payment_installments"],
    "Reviews": ["review_score"]
}

for name, columns in numeric_columns_to_check.items():
    df = datasets[name]
    print("\n" + name)

    for column in columns:
        count = (df[column] < 0).sum()
        print(f"{column:<30}Negative values: {count}")


## 20. Zero Value Check

In [ ]:
zero_columns = {
    "Products": [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ],
    "Payments": ["payment_installments"]
}

for name, columns in zero_columns.items():
    df = datasets[name]
    print("\n" + name)

    for column in columns:
        print(
            f"{column:<30}"
            f"Zero values: {(df[column] == 0).sum()}"
        )


## 21. Items Per Order

In [ ]:
items_per_order = order_items.groupby("order_id").size()

print(items_per_order.value_counts().sort_index())
print("\nSummary:")
display(items_per_order.describe().round(2))


## 22. Master Order-Item Dataset

In [ ]:
order_items_master = (
    order_items
    .merge(
        orders[[
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]],
        on="order_id",
        how="left"
    )
    .merge(
        products[[
            "product_id",
            "product_category_name",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]],
        on="product_id",
        how="left"
    )
    .merge(
        sellers[[
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        ]],
        on="seller_id",
        how="left"
    )
    .merge(
        customers[[
            "customer_id",
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state"
        ]],
        on="customer_id",
        how="left"
    )
    .merge(
        category_translation,
        on="product_category_name",
        how="left"
    )
)

print("Master dataset created.")
print("Shape:", order_items_master.shape)


## 23. Master Dataset Check

In [ ]:
print("Rows:", f"{len(order_items_master):,}")
print("Columns:", order_items_master.shape[1])
display(order_items_master.head())


## 24. Category Label

In [ ]:
order_items_master["category"] = (
    order_items_master["product_category_name_english"]
    .fillna(order_items_master["product_category_name"])
)

print("Category label created.")


## 25. Sales Overview

In [ ]:
total_sales = order_items["price"].sum()
total_freight = order_items["freight_value"].sum()
total_order_value = total_sales + total_freight
total_orders_with_items = order_items["order_id"].nunique()

average_order_value = (
    order_items.groupby("order_id")["price"].sum().mean()
)

print("Total Product Sales:", f"{total_sales:,.2f}")
print("Total Freight:", f"{total_freight:,.2f}")
print("Product + Freight:", f"{total_order_value:,.2f}")
print("Orders with Items:", f"{total_orders_with_items:,}")
print("Average Product Sales / Order:", f"{average_order_value:,.2f}")


## 26. Monthly Business Analysis

In [ ]:
order_items_master["order_month"] = (
    order_items_master["order_purchase_timestamp"].dt.to_period("M")
)

monthly_business = (
    order_items_master
    .groupby("order_month")
    .agg(
        sales=("price", "sum"),
        freight=("freight_value", "sum"),
        orders=("order_id", "nunique"),
        items=("order_id", "size")
    )
)

monthly_business["average_order_value"] = (
    monthly_business["sales"] / monthly_business["orders"]
)

monthly_business["sales_growth_pct"] = (
    monthly_business["sales"].pct_change() * 100
)

display(monthly_business.round(2))


## 27. Yearly Business Analysis

In [ ]:
order_items_master["order_year"] = (
    order_items_master["order_purchase_timestamp"].dt.year
)

yearly_business = (
    order_items_master
    .groupby("order_year")
    .agg(
        sales=("price", "sum"),
        freight=("freight_value", "sum"),
        orders=("order_id", "nunique"),
        items=("order_id", "size")
    )
)

yearly_business["average_order_value"] = (
    yearly_business["sales"] / yearly_business["orders"]
)

yearly_business["sales_growth_pct"] = (
    yearly_business["sales"].pct_change() * 100
)

display(yearly_business.round(2))


## 28. Order Status Analysis

In [ ]:
order_status_summary = (
    orders["order_status"]
    .value_counts()
    .to_frame("orders")
)

order_status_summary["percentage"] = (
    order_status_summary["orders"] / len(orders) * 100
)

display(order_status_summary.round(2))


## 29. Category Sales

In [ ]:
category_sales = (
    order_items_master
    .groupby("category")
    .agg(
        sales=("price", "sum"),
        orders=("order_id", "nunique"),
        items=("order_id", "size")
    )
    .sort_values("sales", ascending=False)
)

category_sales["sales_share_pct"] = (
    category_sales["sales"] / total_sales * 100
)

display(category_sales.head(15).round(2))


## 30. Category Order Volume

In [ ]:
category_orders = (
    order_items_master.groupby("category")["order_id"]
    .nunique()
    .sort_values(ascending=False)
)

display(category_orders.head(15))


## 31. Product Count By Category

In [ ]:
translation_map = category_translation.set_index(
    "product_category_name"
)["product_category_name_english"]

products_with_category = products.assign(
    category=products["product_category_name"]
    .map(translation_map)
    .fillna(products["product_category_name"])
)

category_product_count = (
    products_with_category
    .groupby("category")["product_id"]
    .nunique()
    .sort_values(ascending=False)
)

display(category_product_count.head(15))


## 32. Customer Purchase Frequency

In [ ]:
customer_orders = (
    orders
    .merge(
        customers[["customer_id", "customer_unique_id"]],
        on="customer_id",
        how="left"
    )
    .groupby("customer_unique_id")["order_id"]
    .nunique()
)

display(customer_orders.value_counts().sort_index())


## 33. Repeat Customer Analysis

In [ ]:
repeat_customers = (customer_orders > 1).sum()
total_unique_customers = customer_orders.shape[0]
repeat_customer_rate = (
    repeat_customers / total_unique_customers * 100
)

print("Unique Customers:", f"{total_unique_customers:,}")
print("Repeat Customers:", f"{repeat_customers:,}")
print("Repeat Customer Rate:", f"{repeat_customer_rate:.2f}%")


## 34. Customer Value

In [ ]:
customer_value = (
    order_items_master
    .groupby("customer_unique_id")
    .agg(
        total_spend=("price", "sum"),
        orders=("order_id", "nunique"),
        items=("order_id", "size")
    )
    .sort_values("total_spend", ascending=False)
)

display(customer_value.head(15).round(2))


## 35. Seller Performance

In [ ]:
seller_performance = (
    order_items_master
    .groupby("seller_id")
    .agg(
        sales=("price", "sum"),
        orders=("order_id", "nunique"),
        items=("order_id", "size"),
        average_price=("price", "mean"),
        total_freight=("freight_value", "sum")
    )
    .sort_values("sales", ascending=False)
)

display(seller_performance.head(15).round(2))


## 36. Payment Analysis

In [ ]:
payment_analysis = (
    payments
    .groupby("payment_type")
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_records=("order_id", "size"),
        unique_orders=("order_id", "nunique")
    )
    .sort_values("total_payment_value", ascending=False)
)

payment_analysis["payment_share_pct"] = (
    payment_analysis["total_payment_value"]
    / payments["payment_value"].sum()
    * 100
)

display(payment_analysis.round(2))


## 37. Payment Installments

In [ ]:
installment_analysis = (
    payments
    .groupby("payment_installments")
    .agg(
        payment_records=("order_id", "size"),
        payment_value=("payment_value", "sum")
    )
    .sort_index()
)

display(installment_analysis.round(2))


## 38. Review Analysis

In [ ]:
review_analysis = (
    reviews["review_score"]
    .value_counts()
    .sort_index()
)

display(review_analysis.to_frame("reviews"))

average_review_score = reviews["review_score"].mean()

positive_reviews = reviews["review_score"].isin([4, 5]).sum()
negative_reviews = reviews["review_score"].eq(1).sum()

print("Average Review Score:", round(average_review_score, 2))
print(
    "4-5 Star Reviews:",
    f"{positive_reviews:,}",
    f"({positive_reviews / len(reviews) * 100:.2f}%)"
)
print(
    "1 Star Reviews:",
    f"{negative_reviews:,}",
    f"({negative_reviews / len(reviews) * 100:.2f}%)"
)


## 39. Low Rating Analysis

In [ ]:
low_rating = reviews[reviews["review_score"] <= 2]

print("1-2 Star Reviews:", len(low_rating))
print(
    "Percentage:",
    f"{len(low_rating) / len(reviews) * 100:.2f}%"
)


## 40. Delivery Time

In [ ]:
delivery_data = orders[
    orders["order_delivered_customer_date"].notna()
].copy()

delivery_data["delivery_days"] = (
    delivery_data["order_delivered_customer_date"]
    - delivery_data["order_purchase_timestamp"]
).dt.total_seconds() / 86400

display(delivery_data["delivery_days"].describe().round(2))


## 41. Estimated vs Actual Delivery

In [ ]:
delivery_data["actual_vs_estimated_days"] = (
    delivery_data["order_delivered_customer_date"]
    - delivery_data["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

delivery_data["delivery_result"] = np.where(
    delivery_data["actual_vs_estimated_days"] <= 0,
    "On Time / Early",
    "Late"
)

display(delivery_data["delivery_result"].value_counts().to_frame("orders"))


## 42. Delivery Performance By State

In [ ]:
# customer_state is already available in customers; avoid duplicate-column
# confusion by merging only the required state field.
delivery_state = (
    delivery_data
    .merge(
        customers[["customer_id", "customer_state"]].drop_duplicates(),
        on="customer_id",
        how="left"
    )
    .groupby("customer_state")
    .agg(
        average_delivery_days=("delivery_days", "mean"),
        average_delay_days=("actual_vs_estimated_days", "mean"),
        orders=("order_id", "nunique")
    )
    .sort_values("average_delivery_days", ascending=False)
)

display(delivery_state.round(2))


## 43. Freight Analysis

In [ ]:
freight_summary = (
    order_items[["price", "freight_value"]]
    .describe()
    .round(2)
)

price_freight_correlation = (
    order_items[["price", "freight_value"]]
    .corr()
    .loc["price", "freight_value"]
)

display(freight_summary)
print("Price-Freight Correlation:", round(price_freight_correlation, 3))


## 44. Price Outlier Analysis

In [ ]:
price_q1 = order_items["price"].quantile(0.25)
price_q3 = order_items["price"].quantile(0.75)
price_iqr = price_q3 - price_q1
price_upper = price_q3 + 1.5 * price_iqr

price_outliers = order_items[order_items["price"] > price_upper]

print("Q1:", round(price_q1, 2))
print("Q3:", round(price_q3, 2))
print("IQR:", round(price_iqr, 2))
print("Upper Bound:", round(price_upper, 2))
print("Outlier Rows:", f"{len(price_outliers):,}")
print(
    "Outlier Percentage:",
    f"{len(price_outliers) / len(order_items) * 100:.2f}%"
)


## 45. Freight Outlier Analysis

In [ ]:
freight_q1 = order_items["freight_value"].quantile(0.25)
freight_q3 = order_items["freight_value"].quantile(0.75)
freight_iqr = freight_q3 - freight_q1
freight_upper = freight_q3 + 1.5 * freight_iqr

freight_outliers = order_items[
    order_items["freight_value"] > freight_upper
]

print("Q1:", round(freight_q1, 2))
print("Q3:", round(freight_q3, 2))
print("IQR:", round(freight_iqr, 2))
print("Upper Bound:", round(freight_upper, 2))
print("Outlier Rows:", f"{len(freight_outliers):,}")
print(
    "Outlier Percentage:",
    f"{len(freight_outliers) / len(order_items) * 100:.2f}%"
)


## 46. Customer State Sales

In [ ]:
customer_state_sales = (
    order_items_master
    .groupby("customer_state")
    .agg(
        sales=("price", "sum"),
        orders=("order_id", "nunique"),
        items=("order_id", "size")
    )
    .sort_values("sales", ascending=False)
)

customer_state_sales["sales_share_pct"] = (
    customer_state_sales["sales"] / total_sales * 100
)

display(customer_state_sales.round(2))


## 47. Seller State Sales

In [ ]:
seller_state_sales = (
    order_items_master
    .groupby("seller_state")
    .agg(
        sales=("price", "sum"),
        orders=("order_id", "nunique"),
        items=("order_id", "size")
    )
    .sort_values("sales", ascending=False)
)

display(seller_state_sales.round(2))


## 48. Customer City Analysis

In [ ]:
customer_city_analysis = (
    customers["customer_city"]
    .value_counts()
    .head(15)
)

display(customer_city_analysis.to_frame("customers"))


## 49. Seller City Analysis

In [ ]:
seller_city_analysis = (
    sellers["seller_city"]
    .value_counts()
    .head(15)
)

display(seller_city_analysis.to_frame("sellers"))


## 50. Weight vs Freight Correlation

In [ ]:
weight_freight_correlation = (
    order_items_master[["product_weight_g", "freight_value"]]
    .corr()
    .loc["product_weight_g", "freight_value"]
)

print("Weight-Freight Correlation:", round(weight_freight_correlation, 3))


## 51. Product Dimensions vs Freight

In [ ]:
dimension_columns = [
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "freight_value"
]

dimension_correlation = (
    order_items_master[dimension_columns].corr()
)

display(dimension_correlation.round(3))


## 52. Referential Integrity

In [ ]:
missing_order_customers = (
    ~orders["customer_id"].isin(customers["customer_id"])
).sum()

missing_products = (
    ~order_items["product_id"].isin(products["product_id"])
).sum()

missing_sellers = (
    ~order_items["seller_id"].isin(sellers["seller_id"])
).sum()

missing_orders = (
    ~order_items["order_id"].isin(orders["order_id"])
).sum()

missing_payment_orders = (
    ~payments["order_id"].isin(orders["order_id"])
).sum()

missing_review_orders = (
    ~reviews["order_id"].isin(orders["order_id"])
).sum()

print("Orders → Customers missing:", missing_order_customers)
print("Order Items → Products missing:", missing_products)
print("Order Items → Sellers missing:", missing_sellers)
print("Order Items → Orders missing:", missing_orders)
print("Payments → Orders missing:", missing_payment_orders)
print("Reviews → Orders missing:", missing_review_orders)


## 53. Orders Without Items

In [ ]:
orders_without_items = ~orders["order_id"].isin(order_items["order_id"])
orders_without_items_df = orders[orders_without_items]

print("Orders without items:", len(orders_without_items_df))
print("\nStatus distribution:")
display(
    orders_without_items_df["order_status"]
    .value_counts()
    .to_frame("orders")
)


## 54. Orders Without Payments

In [ ]:
orders_without_payments = ~orders["order_id"].isin(payments["order_id"])
orders_without_payments_df = orders[orders_without_payments]

print("Orders without payments:", len(orders_without_payments_df))

if len(orders_without_payments_df) > 0:
    display(
        orders_without_payments_df[
            ["order_id", "order_status", "order_purchase_timestamp"]
        ]
    )


## 55. Customer State Distribution

In [ ]:
customer_state_distribution = (
    customers["customer_state"]
    .value_counts()
    .to_frame("customers")
)

customer_state_distribution["percentage"] = (
    customer_state_distribution["customers"] / len(customers) * 100
)

display(customer_state_distribution.round(2))


## 56. Seller State Distribution

In [ ]:
seller_state_distribution = (
    sellers["seller_state"]
    .value_counts()
    .to_frame("sellers")
)

seller_state_distribution["percentage"] = (
    seller_state_distribution["sellers"] / len(sellers) * 100
)

display(seller_state_distribution.round(2))


## 57. Category Translation Check

In [ ]:
translation_keys = (
    category_translation["product_category_name"].nunique()
)

translated_categories = (
    category_translation["product_category_name_english"].nunique()
)

product_categories = (
    products["product_category_name"].nunique()
)

untranslated_categories = (
    products["product_category_name"]
    .dropna()
    .loc[
        lambda x: ~x.isin(
            category_translation["product_category_name"]
        )
    ]
    .unique()
)

print("Product categories:", product_categories)
print("Translation keys:", translation_keys)
print("Translated English categories:", translated_categories)
print("Untranslated categories:", len(untranslated_categories))
print("Untranslated:", untranslated_categories)


## 58. Final KPI Summary

In [ ]:
average_delivery_days = delivery_data["delivery_days"].mean()

print("=" * 80)
print("FINAL KPI SUMMARY")
print("=" * 80)

print(f"{'Total Product Sales':<35}: {total_sales:,.2f}")
print(f"{'Total Freight':<35}: {total_freight:,.2f}")
print(f"{'Product + Freight':<35}: {total_order_value:,.2f}")
print(f"{'Orders with Items':<35}: {total_orders_with_items:,}")
print(f"{'Average Order Value':<35}: {average_order_value:,.2f}")
print(f"{'Unique Customers':<35}: {total_unique_customers:,}")
print(f"{'Repeat Customer Rate':<35}: {repeat_customer_rate:.2f}%")
print(f"{'Average Review Score':<35}: {average_review_score:.2f}")
print(f"{'Average Delivery Days':<35}: {average_delivery_days:.2f}")
print(f"{'Price-Freight Correlation':<35}: {price_freight_correlation:.3f}")
print(f"{'Weight-Freight Correlation':<35}: {weight_freight_correlation:.3f}")


## 59. Visualizations

The following cells generate the major portfolio-level EDA charts.

In [ ]:
# Monthly Sales

plt.figure(figsize=(12, 5))
monthly_business["sales"].plot()
plt.title("Monthly Product Sales")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Monthly Orders

plt.figure(figsize=(12, 5))
monthly_business["orders"].plot()
plt.title("Monthly Orders")
plt.xlabel("Month")
plt.ylabel("Orders")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Order Status

plt.figure(figsize=(10, 5))
orders["order_status"].value_counts().plot(kind="bar")
plt.title("Order Status Distribution")
plt.xlabel("Order Status")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Top Categories

plt.figure(figsize=(10, 6))
category_sales["sales"].head(10).sort_values().plot(kind="barh")
plt.title("Top 10 Product Categories by Sales")
plt.xlabel("Sales")
plt.ylabel("Category")
plt.tight_layout()
plt.show()


In [ ]:
# Review Scores

plt.figure(figsize=(8, 5))
reviews["review_score"].value_counts().sort_index().plot(kind="bar")
plt.title("Review Score Distribution")
plt.xlabel("Review Score")
plt.ylabel("Number of Reviews")
plt.tight_layout()
plt.show()


In [ ]:
# Payment Types

plt.figure(figsize=(8, 5))
payments["payment_type"].value_counts().plot(kind="bar")
plt.title("Payment Type Distribution")
plt.xlabel("Payment Type")
plt.ylabel("Payment Records")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Customer States

plt.figure(figsize=(10, 6))
customer_state_distribution["customers"].head(10).sort_values().plot(kind="barh")
plt.title("Top 10 Customer States")
plt.xlabel("Customers")
plt.ylabel("State")
plt.tight_layout()
plt.show()


In [ ]:
# Seller States

plt.figure(figsize=(10, 6))
seller_state_distribution["sellers"].head(10).sort_values().plot(kind="barh")
plt.title("Top 10 Seller States")
plt.xlabel("Sellers")
plt.ylabel("State")
plt.tight_layout()
plt.show()


In [ ]:
# Price Distribution

plt.figure(figsize=(10, 5))
sns.histplot(
    order_items["price"],
    bins=50,
    kde=True
)
plt.title("Product Price Distribution")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
# Freight Distribution

plt.figure(figsize=(10, 5))
sns.histplot(
    order_items["freight_value"],
    bins=50,
    kde=True
)
plt.title("Freight Value Distribution")
plt.xlabel("Freight Value")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
# Price vs Freight

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=order_items,
    x="price",
    y="freight_value",
    alpha=0.4
)
plt.title("Product Price vs Freight Value")
plt.xlabel("Product Price")
plt.ylabel("Freight Value")
plt.tight_layout()
plt.show()


In [ ]:
# Delivery Time Distribution

plt.figure(figsize=(10, 5))
sns.histplot(
    delivery_data["delivery_days"],
    bins=50,
    kde=True
)
plt.title("Delivery Time Distribution")
plt.xlabel("Delivery Days")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
# Review Score vs Delivery Time

review_delivery = (
    reviews[["order_id", "review_score"]]
    .merge(
        orders[[
            "order_id",
            "order_delivered_customer_date",
            "order_purchase_timestamp"
        ]],
        on="order_id",
        how="left"
    )
)

review_delivery = review_delivery[
    review_delivery["order_delivered_customer_date"].notna()
].copy()

review_delivery["delivery_days"] = (
    review_delivery["order_delivered_customer_date"]
    - review_delivery["order_purchase_timestamp"]
).dt.total_seconds() / 86400

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=review_delivery,
    x="review_score",
    y="delivery_days"
)
plt.title("Review Score vs Delivery Time")
plt.xlabel("Review Score")
plt.ylabel("Delivery Days")
plt.tight_layout()
plt.show()


# 60. EDA Completed

All major EDA and business-analysis sections have been executed.

### Next Step
Interpret the outputs and convert the strongest findings into:
- Business insights
- Recommendations
- Power BI dashboard KPIs
- Portfolio/project conclusions
